In [ ]:
from google.colab import files
uploaded = files.upload()

Saving test_unsupervised.csv to test_unsupervised.csv
Saving train_unsupervised.csv to train_unsupervised.csv


In [ ]:
!pip install seaborn

In [ ]:
import pandas as pd

train = pd.read_csv("/content/train_unsupervised.csv")
test  = pd.read_csv("/content/test_unsupervised.csv")

X_train = train
X_test  = test.drop("label", axis=1)
y_test  = test["label"]

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

Model-1: Isolation Forest


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score

best_model = None
best_f1 = 0

for n in [200, 300, 400]:
    for c in [0.1, 0.15, 0.2]:   # IMPORTANT: increase contamination
        for depth in [None, 10, 15]:

            model = IsolationForest(
                n_estimators=n,
                contamination=c,
                max_samples="auto",
                max_features=0.8,
                random_state=42,
                n_jobs=-1
            )

            model.fit(X_train)
            pred = (model.predict(X_test) == -1).astype(int)

            f1 = f1_score(y_test, pred)

            if f1 > best_f1:
                best_f1 = f1
                best_model = model

print("Best F1:", best_f1)

pred_iso = (best_model.predict(X_test) == -1).astype(int)

Best F1: 0.5678155219439623


Model-2: One-class SVM

In [ ]:
from sklearn.svm import OneClassSVM

best_f1 = 0
best_model = None

for nu in [0.03, 0.05, 0.1]:
    for gamma in ["scale", "auto"]:

        model = OneClassSVM(
            kernel="rbf",
            nu=nu,
            gamma=gamma
        )

        model.fit(X_train)
        pred = model.predict(X_test)
        pred = (pred == -1).astype(int)

        f1 = f1_score(y_test, pred)

        if f1 > best_f1:
            best_f1 = f1
            best_model = model

print("Best OC-SVM F1:", best_f1)

pred_svm = (best_model.predict(X_test) == -1).astype(int)

Best OC-SVM F1: 0.33537414965986395


In [ ]:
from sklearn.metrics import classification_report

print("Isolation Forest")
print(classification_report(y_test, pred_iso))

print("\nOne-Class SVM")
print(classification_report(y_test, pred_svm))

Isolation Forest
              precision    recall  f1-score   support

           0       0.82      0.81      0.81      4656
           1       0.56      0.57      0.57      1998

    accuracy                           0.74      6654
   macro avg       0.69      0.69      0.69      6654
weighted avg       0.74      0.74      0.74      6654


One-Class SVM
              precision    recall  f1-score   support

           0       0.74      0.90      0.81      4656
           1       0.52      0.25      0.34      1998

    accuracy                           0.71      6654
   macro avg       0.63      0.58      0.57      6654
weighted avg       0.67      0.71      0.67      6654



In [ ]:
import pandas as pd

pred_series = pd.Series(pred)

# require persistence (like MAD)
pred_smooth = (
    pred_series.rolling(2)
    .sum()
    .ge(2)
    .astype(int)
)